# Robustesse B/C : requêtes bruitées à 5 % et 10 %
Environnement CPU, sur le même Drive que les 38 morceaux du corpus. Peut tourner pendant Eos.
Importer `search_inputs_noise.json.gz` dans le volet Fichiers, puis Tout exécuter.
Les 250 requêtes utilisent les textes et masques figés sur le PC. Aucun jugement transmis.
Même modèle, révision, float32, préfixe et recherche exacte que le propre.
Les sorties sont distinctes du propre ; le carnet ne modifie pas les embeddings.


In [ ]:
# 1. GPU et Google Drive (compte Pro : c'est là que sont les 38 morceaux)
import os, subprocess
import torch
DEVICE = "cpu"  # le GPU n'est pas nécessaire : 250 requêtes seulement
print("calcul sur :", DEVICE)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
EMB = Path("/content/drive/MyDrive/hybrid_ir/exp1_embeddings")
chunks = sorted(EMB.glob("chunk_*.npy"))
print(len(chunks), "morceaux trouvés")
assert len(chunks) == 38, "il faut les 38 morceaux : vérifier que Drive est celui du compte Pro"

In [ ]:
# 2. Bibliothèques aux versions de l'encodage
!pip install -q "sentence-transformers==6.2.0" "transformers==5.22.1"
import sentence_transformers, transformers
print(sentence_transformers.__version__, transformers.__version__)

In [ ]:
# Entrées bruitées, empreinte figée avant toute évaluation
import gzip, json, hashlib, base64
import numpy as np
from google.colab import files
from pathlib import Path
path = Path('/content/search_inputs_noise.json.gz')
if not path.exists():
    files.upload()
blob = path.read_bytes()
INPUT_SHA256 = '1552c092cfbfca169bb21710a361e91e81b87d0384b13aefcbb815c842726a62'
assert hashlib.sha256(blob).hexdigest() == INPUT_SHA256, 'Entrées différentes du protocole figé'
inputs = json.loads(gzip.decompress(blob))
assert len(inputs['queries']) == 250
print(len(inputs['queries']), 'requêtes bruitées ;', inputs['noise_sha256'])


In [ ]:
# 4. Vecteurs des essais, dans l'ordre du PC (vérifié par empreinte)
doc_ids = json.loads((EMB / "doc_ids.json").read_text())
assert hashlib.sha256("\n".join(doc_ids).encode()).hexdigest() == inputs["doc_ids_sha256"], "ordre des essais différent du PC : prévenir"
E = np.concatenate([np.load(c) for c in chunks])
assert E.shape == (inputs["n_docs"], 768), E.shape
norms = np.linalg.norm(E.astype(np.float32), axis=1)
print(E.shape, "| normes min %.4f max %.4f" % (norms.min(), norms.max()))
E_dev = torch.from_numpy(E).to(DEVICE).float()

In [ ]:
# 5. Encodage des requêtes : même modèle, même révision, float32, gabarit « requête » de la fiche
from sentence_transformers import SentenceTransformer
manifest = json.loads((EMB / "manifest.json").read_text())
model = SentenceTransformer(manifest["model"], revision=manifest["revision"], device=DEVICE,
                            model_kwargs={"torch_dtype": torch.float32}, config_kwargs={"vision_config": None, "audio_config": None})
query_texts = ["task: search result | query: " + " ".join(q["text"].split()) for q in inputs["queries"]]
Q = model.encode(query_texts, batch_size=8, normalize_embeddings=True, convert_to_numpy=True)
assert not np.isnan(Q).any()
print(Q.shape)

In [ ]:
# 6. Recherche : 1 000 premiers essais ; égalités départagées par l'ordre du corpus
TOP_K = 1000
def top_ids(scores):
    candidates = np.argpartition(-scores, TOP_K)[:TOP_K] if len(scores) > TOP_K else np.arange(len(scores))
    candidates = candidates[np.isfinite(scores[candidates])]
    ranked = candidates[np.lexsort((candidates, -scores[candidates]))]
    return [doc_ids[i] for i in ranked]
rankings = {"B": {}, "C": {}}
for q, vector in zip(inputs["queries"], Q):
    scores = (E_dev @ torch.from_numpy(vector).to(DEVICE)).cpu().numpy().astype(np.float64)
    mask = np.unpackbits(np.frombuffer(base64.b64decode(q["kept_mask"]), dtype=np.uint8), count=inputs["n_docs"]).astype(bool)
    assert mask.sum() == q["n_kept"]
    key = f"{q['year']}/{q['query_id']}"
    rankings["B"][key] = top_ids(scores)
    rankings["C"][key] = top_ids(np.where(mask, scores, -np.inf))
print(len(rankings["B"]), "requêtes bruitées")

In [ ]:
# 7. Sauvegarde sur Drive et téléchargement du résultat (≈ 1 Mo)
result = {"manifest": manifest, "top_k": TOP_K, "doc_ids_sha256": inputs["doc_ids_sha256"],
          "input_sha256": INPUT_SHA256, "noise_sha256": inputs["noise_sha256"], "query_device": DEVICE, "query_dtype": str(Q.dtype), "query_versions": {"torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__}, "query_prefix": "task: search result | query: ", "rankings": rankings}
blob = gzip.compress(json.dumps(result).encode())
(EMB / "exp1_rankings_bc_noise.json.gz").write_bytes(blob)
open("exp1_rankings_bc_noise.json.gz", "wb").write(blob)
files.download("exp1_rankings_bc_noise.json.gz")
print("TERMINÉ", len(blob) // 1024, "Ko")